### **Setup: Imports und Initialisierung**

In [1]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath('..'))

from quinex_application.mongos_store import MongoStore
from quinex_application.quinex import Quinex
from quinex_application.visualisation_quinex import VisualisationQuinex
from quinex_application.oeo_data import OEOData
from quinex_application.levenshtein import LevenshteinDistance
from quinex_application.techdb import TechDB

In [4]:
quinex = Quinex()
mongodb = MongoStore(quinex, database_name="quinex_db_query4", collection_name="papers_2025")
visualisation = VisualisationQuinex()
oeo = OEOData("../data/oeo/etd.xlsx", "../data/oeo/oeo-full.owl", "../data/oeo/oeo.xlsx")
levenshtein = LevenshteinDistance()
tb = TechDB()

### **Load Analysis into DataFrame**

In [ ]:
analysis = mongodb.get_collection("papers_2024")
df_path = "../data/quinex/df_quinex.parquet"

df_quinex = visualisation.create_df(analysis)
visualisation.save_df(df_quinex, df_path)

display(df_quinex.head(5))
print(len(df_quinex))

In [5]:
import pandas as pd
df_quinex = pd.read_parquet("../data/quinex/df_quinex.parquet")
print(len(df_quinex))
print(len(mongodb.get_collection("papers_2025")))
print(len(mongodb.get_collection("papers_2024")))

280926
562
2296


### **Metadata Results**

In [ ]:
# Alle Paper-IDs, die in analysis vorkommen (mit Analyse)
analysis_ids = {paper["_id"] for paper in analysis if paper.get("analysis_result")}

# Alle Paper-IDs, die tatsächlich im DataFrame gelandet sind
df_ids = set(df_quinex["paper_id"].unique())

# Fehlende Paper mit Analyse, aber ohne quantitative Aussage
missing = analysis_ids - df_ids

print(f"🧾 Total analyzed papers (with analysis_result): {len(analysis_ids)}")
print(f"✅ Papers in DataFrame: {len(df_ids)}")
print(f"❌ Papers with no valid quantitative statements: {len(missing)}")

# Optional: welche das sind
print("Missing paper_ids:", missing)

In [ ]:
# Gesamtanzahl der quantitativen Aussagen
total_statements = len(df_quinex)

# Durchschnittliche Anzahl an Aussagen pro Paper
avg_statements_per_paper = total_statements / len(df_ids)

print(f"📊 Total quantitative statements: {total_statements}")
print(f"➗ Average statements per paper: {avg_statements_per_paper:.2f}")

### **Apply Levenshtein Method with TechDB Hirarchy**

In [3]:
en_list = tb.save_entities()
prop_list = tb.save_properties()

In [4]:
print(en_list)

['artificial object', 'transport network component', 'tunnel', 'railway', 'bridge', 'transport hub', 'port', 'passenger port', 'freight port', 'bus station', 'train station', 'freight train station', 'passenger train station', 'airport', 'canal', 'road', 'dam', 'hardware', 'building', 'tower', 'vehicle', 'internal combustion vehicle', 'gas vehicle', 'liquefied petroleum gas vehicle', 'liquefied natural gas vehicle', 'compressed gas vehicle', 'gasoline vehicle', 'diesel vehicle', 'watercraft', 'ship', 'cargo ship', 'container ship', 'tank ship', 'passenger ship', 'land vehicle', 'train', 'passenger train', 'freight train', 'road vehicle', 'bicycle', 'e-bike', 'motorised road vehicle', 'truck', 'liquefied natural gas truck', 'fuel cell electric truck', 'battery electric truck', 'compressed gas truck', 'diesel truck', 'bus', 'motorcycle', 'car', 'fuel cell electric car', 'plug-in hybrid electric car', 'compressed gas car', 'gasoline car', 'battery electric car', 'diesel car', 'liquefied p

In [ ]:
df_path = "../data/quinex/df_quinex_levenshtein.parquet"
df_quinex = visualisation.load_df("../data/quinex/df_quinex.parquet")

df_quinex[['entity_mapping', 'entity_match_score']] = df_quinex['entity'].apply(lambda x: pd.Series(levenshtein.get_best_match(x, en_list, "similarity")))

In [ ]:
df_quinex[['property_mapping', 'property_match_score']] = df_quinex['property'].apply(lambda x: pd.Series(levenshtein.get_best_match(x, prop_list, "similarity")))
visualisation.save_df(df_quinex, df_path)

In [ ]:
display(df_quinex.head(1))

In [ ]:
df_levenshtein = df_quinex[(df_quinex['entity_match_score'] >= 0.75) & (df_quinex['property_match_score'] >=0.75)]
display(df_levenshtein.head(10))
print(len(df_levenshtein))

In [ ]:
visualisation.save_df(df_levenshtein, "../data/quinex/df_quinex_prop_ent_mapped.parquet")

In [ ]:
print(df_levenshtein["entity_mapping"].nunique())
print(df_levenshtein["property_mapping"].nunique())

### **Visualise Results**

In [ ]:
fig = visualisation.plot_top_values(df_quinex, column="entity", top_n=25, ignore_empty=True, figsize=(6,6))
fig = visualisation.plot_top_values(df_quinex, column="entity", top_n=25, match_column="best_oeo_match", score_column="match_score", figsize=(6,6))

In [ ]:
fig = visualisation.plot_top_properties_of_entity(df_quinex, "electrolyzer", top_n=10, ignore_empty=True, figsize=(6,6))
fig = visualisation.plot_top_quantity_of_property(df_quinex, "electrolyzer", "rated power", top_n=8, ignore_empty=True, figsize=(6,6))

In [ ]:
df_path = "../data/quinex/df_quinex_levenshtein.parquet"
fig = visualisation.plot_levenshtein_score_limits(df_path, score_limits=[0.7, 0.75, 0.8, 0.85, 0.9])
fig = visualisation.plot_unique_entities_over_score_limits(df_path, score_limits=[0.7, 0.75, 0.8, 0.85, 0.9])